In [1]:
#from torch import nn
import pandas as pd
import numpy as np
from pathlib import Path
import torch
from nfl.lib import enums

from nfl.data_management.DataManager import DataManager
from nfl.NeuralNetwork.EPA_Predictor import EPAPredictor
from nfl.NeuralNetwork.NNSolver import Solver
from torch.utils.tensorboard import SummaryWriter

# TensorBoard writer setup
writer = SummaryWriter(log_dir="runs/epa_hyperparameter_sweep")

In [2]:
data_path = (Path.cwd() / "nfl/data").resolve()

In [3]:
data = DataManager.get_data(path_to_json = data_path)

reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2015.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2005.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2016.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2006.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_1999.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2023.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2008.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2014.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2017.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2009.csv
reading /home/peter/personal/my_machine_learning/nfl_analyti

In [4]:
EXCLUDED_PLAY_TYPES = {
    enums.PlayType.KICK,
    enums.PlayType.EXTRA_POINT,
    enums.PlayType.NO_PLAY,
    enums.PlayType.GAME_START,
}

data = data[
            data["play_type"].notna()   
            & ~data["play_type"].isin(EXCLUDED_PLAY_TYPES)]
pd.set_option('display.max_columns', None)

data["has_roof"] = data["roof"].isin(
    [enums.RoofType.DOME, enums.RoofType.CLOSED]
)

data["has_turf"] = data["surface"] != enums.SurfaceType.GRASS

gap_map = {"guard": 1, "tackle": 2, "end": 3}
data["run_gap"] = data["run_gap"].map(gap_map).fillna(-5).astype(int)

data.head(5)

,yardline_100,game_seconds_remaining,quarter_end,drive,sp,qtr,down,goal_to_go,ydstogo,ydsnet,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,air_yards,yards_after_catch,run_gap,field_goal_result,extra_point_result,score_differential,score_differential_post,ep,epa,wp,home_wp,wpa,home_wp_post,air_wpa,yac_wpa,comp_air_wpa,comp_yac_wpa,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,fourth_down_converted,fourth_down_failed,incomplete_pass,touchback,interception,fumble_forced,fumble_not_forced,fumble_out_of_bounds,solo_tackle,safety,penalty,tackled_for_loss,fumble_lost,qb_hit,rush_attempt,pass_attempt,sack,touchdown,pass_touchdown,rush_touchdown,return_touchdown,extra_point_attempt,field_goal_attempt,fumble,complete_pass,assist_tackle,passing_yards,receiving_yards,rushing_yards,tackle_with_assist,fumble_recovery_1_yards,fumble_recovery_2_yards,return_yards,penalty_yards,replay_or_challenge,replay_or_challenge_result,penalty_type,season,cp,cpoe,series,series_success,series_result,order_sequence,play_clock,play_type_nfl,st_play_type,end_yard_line,drive_play_count,drive_first_downs,drive_ended_with_score,drive_quarter_start,drive_quarter_end,drive_yards_penalized,drive_start_transition,drive_end_transition,drive_game_clock_start,drive_game_clock_end,drive_start_yard_line,drive_end_yard_line,result,div_game,roof,surface,temp,wind,aborted_play,success,pass,rush,first_down,special,play,out_of_bounds,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe,day_of_season,has_roof,has_turf
2,80.0,3600.0,False,1.0,False,1,1.0,False,10,18.0,pass,3.0,False,False,True,False,False,False,short,3.0,0.0,-5,NaN,NaN,0.0,0.0,0.239785,-0.337139,0.422024,0.577976,-0.001425,0.579401,-0.001425,0.000000,-0.001425,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,0.0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,3.0,3.0,NaN,False,NaN,NaN,0.0,NaN,False,NaN,NaN,2015,0.765811,23.418927,1,True,First down,51.0,12.0,PASS,NaN,23.0,6.0,1.0,False,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,False,outdoors,grass,88.0,13.0,False,False,True,False,False,False,True,False,4.699278,3.0,0.678964,0.225919,0.456481,54.351911,43,False,False
3,77.0,3573.0,False,1.0,False,1,2.0,False,7,18.0,run,2.0,False,False,False,False,False,False,NaN,NaN,NaN,1,NaN,NaN,0.0,0.0,-0.097354,-0.262481,0.420599,0.579401,-0.017304,0.596705,NaN,NaN,0.000000,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,0.0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN,2.0,False,NaN,NaN,0.0,NaN,False,NaN,NaN,2015,NaN,NaN,1,True,First down,75.0,18.0,RUSH,NaN,25.0,6.0,1.0,False,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,False,outdoors,grass,88.0,13.0,False,False,False,True,False,False,True,False,NaN,NaN,NaN,NaN,0.545905,-54.590458,43,False,False
4,75.0,3532.0,False,1.0,False,1,3.0,False,5,18.0,pass,10.0,True,False,True,False,False,False,short,4.0,6.0,-5,NaN,NaN,0.0,0.0,-0.359835,1.661242,0.403295,0.596705,0.045358,0.551347,0.000000,0.045358,0.000000,0.045358,False,True,False,True,False,False,False,False,False,False,False,False,False,True,0.0,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,10.0,10.0,NaN,False,NaN,NaN,0.0,NaN,False,NaN,NaN,2015,0.510176,48.982388,1,True,First down,96.0,5.0,PASS,NaN,35.0,6.0,1.0,False,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,False,outdoors,grass,88.0,13.0,False,True,True,False,True,False,True,False,4.662650,2.0,0.712350,0.712350,0.968533,3.146732,43,False,False
5,65.0,3494.0,False,1.0,False,1,1.0,False,10,18.0,run,0.0,False,False,False,False,False,False,NaN,NaN,NaN,2,NaN,NaN,0.0,0.0,1.301407,-0.518931,0.448653,0.551347,-0.018066,0.569413,NaN,NaN,0.000000,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,F

In [5]:
scenario_columns = [
    "yardline_100",
    "game_seconds_remaining",
    "has_turf",
    "temp",
    "wind",
    "has_roof",
    "ydstogo",
    "goal_to_go",
    "score_differential",
    "down",
    "div_game",
    "day_of_season",
    "series"
]

play_columns = [
    "play_type",
    "pass_location",
#    "pass_length",
#    "run_location",
    "run_gap",
    "shotgun",
    "no_huddle",
    "qb_kneel",
    "qb_spike",
    "qb_scramble",
    "air_yards"
]

result_columns = [
    "epa",
    "wpa",
    "success",
    "result",
    "series_success",
    "tackle_for_loss",
    "saftey",
    "yards_gained",
    "touchdown",
    "fumble",
    "complete_pass",
    "rushing_yards",
    "fumble_lost",
    "interception",
    "sack",
    "penalty_yards",
]

In [6]:
data = pd.get_dummies(data, columns=["play_type"], dtype=float)
epa_feature_columns = []
for col in data.columns:
    if (
        col in scenario_columns
        or col in play_columns
        or col.startswith("play_type_") # all the one hot endoded from play_type
    ):
        if col != "play_type" and col != "play_type_nfl":
            epa_feature_columns.append(col)

# convert NaN to 0/False            
data[epa_feature_columns] = (
    data[epa_feature_columns].astype(float).fillna(0.0)
)

In [7]:
# 3. Split and convert to tensors as usual
train_percentage = 0.8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_data = data.sample(frac=train_percentage, random_state=42)
test_data = data.drop(train_data.index)

X_train = torch.tensor(
    train_data[epa_feature_columns].values, dtype=torch.float32
).to(device)
y_train = torch.tensor(train_data["epa"].values, dtype=torch.float32).to(
    device
)

x_test = torch.tensor(
    test_data[epa_feature_columns].values, dtype=torch.float32
).to(device)
y_test = torch.tensor(test_data["epa"].values, dtype=torch.float32).to(device)

In [8]:
print("Cuda device count: ", torch.cuda.device_count())
print(f"Using {device} device")

EPA_INPUT_SIZE = len(epa_feature_columns)
NUM_ITER = 7
NUM_EPOCH = 100
EPS = 1e-8
BETAS = (0.9, 0.999)


MIN_HIDDEN_LAYERS, MAX_HIDDEN_LAYERS = 1, 20
MIN_HIDDEN_SIZE, MAX_HIDDEN_SIZE = 16, 256
MIN_LEARNING_RATE, MAX_LEARNING_RATE = 1e-5, 1e-2
MIN_BATCH_SIZE, MAX_BATCH_SIZE = 64, 512
MIN_WEIGHT_DECAY, MAX_WEIGHT_DECAY = 1e-5, 1e-2

BATCH_SIZE = np.linspace(MIN_BATCH_SIZE, MAX_BATCH_SIZE, num=NUM_ITER).astype(int).tolist()
LEARNING_RATE = np.geomspace(MIN_LEARNING_RATE, MAX_LEARNING_RATE, num=NUM_ITER).tolist()
NUM_HIDDEN_LAYERS = np.linspace(MIN_HIDDEN_LAYERS, MAX_HIDDEN_LAYERS, num=NUM_ITER).astype(int).tolist()
HIDDEN_SIZE = np.linspace(MIN_HIDDEN_SIZE, MAX_HIDDEN_SIZE, num=NUM_ITER).astype(int).tolist()
WEIGHT_DECAY = np.geomspace(MIN_WEIGHT_DECAY, MAX_WEIGHT_DECAY, num=NUM_ITER).tolist()

Cuda device count:  1
Using cuda device


In [ ]:
writer = SummaryWriter(log_dir="runs/epa_hyperparameter_sweep")

for bs in BATCH_SIZE:
    for lr in LEARNING_RATE:
        for hl in NUM_HIDDEN_LAYERS:
            for hs in HIDDEN_SIZE:
                for wd in WEIGHT_DECAY:

                    run_name = f"bs{bs}_lr{lr:.1e}_hl{hl}_hs{hs}_wd{wd:.1e}"
                    hparams = {
                        "batch_size": int(bs),
                        "lr": float(lr),
                        "num_hidden_layers": int(hl),
                        "hidden_size": int(hs),
                        "weight_decay": float(wd),
                    }

                    model = EPAPredictor(
                        input_size=EPA_INPUT_SIZE,
                        num_hidden_layers=int(hl),
                        hidden_size=int(hs),
                    ).to(device)

                    optimizer = torch.optim.Adam(
                        model.parameters(),
                        lr=float(lr),
                        betas=BETAS,
                        eps=EPS,
                        weight_decay=float(wd),
                    )

                    solver = Solver(
                        model=model,
                        device=device,
                        num_epochs=NUM_EPOCH,
                        batch_size=bs,
                        optimizer=optimizer,
                        criterion=torch.nn.MSELoss().to(device),
                        writer=writer,
                        run_name=run_name,
                        hparams=hparams,
                    )

                    solver.train(
                        X_train,
                        y_train,
                        x_test,
                        y_test,
                        modelName=run_name,
                        saveBest=True,
                    )

writer.close()

/home/peter/personal/my_machine_learning/nfl_analytics/nfl/NeuralNetwork/NNSolver.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(x, dtype=torch.float32),
/home/peter/personal/my_machine_learning/nfl_analytics/nfl/NeuralNetwork/NNSolver.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(y, dtype=torch.float32),
/home/peter/personal/my_machine_learning/nfl_analytics/nfl/NeuralNetwork/NNSolver.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(x_v, dtype=torch.float32),
/home/peter/personal/my

Epoch   0/100 | Train Loss: nan | Valid Loss: nan
